In [1]:
!git clone https://github.com/mazinbashierrr/Meeting-Accountability-Predictor.git
%cd Meeting-Accountability-Predictor
!ls

Cloning into 'Meeting-Accountability-Predictor'...
remote: Enumerating objects: 2534, done.
remote: Counting objects: 100% (2534/2534), done.
remote: Compressing objects: 100% (970/970), done.
remote: Total 2534 (delta 1563), reused 2530 (delta 1562), pack-reused 0 (from 0)
Receiving objects: 100% (2534/2534), 23.96 MiB | 7.74 MiB/s, done.
Resolving deltas: 100% (1563/1563), done.
/content/Meeting-Accountability-Predictor
data				      roberta_evaluation_results.json
meeting_accountability.ipynb	      splits
README.md			      utterances.parquet
roberta_action_item_classifier.ipynb


In [2]:
#Load Dataset
import pandas as pd

train_df = pd.read_json("splits/train.jsonl", lines=True)
dev_df = pd.read_json("splits/dev.jsonl", lines=True)
test_df = pd.read_json("splits/test.jsonl", lines=True)

print("Train:", train_df.shape)
print("Dev:", dev_df.shape)
print("Test:", test_df.shape)

Train: (81080, 4)
Dev: (17401, 4)
Test: (16907, 4)


In [3]:
print(train_df.columns)

train_df.head()

Index(['text', 'label', 'meeting', 'speaker'], dtype='object')


,text,label,meeting,speaker
0,"Hi, I'm David and I'm supposed to be an indust...",0,ES2002,A
1,"Um, I just got the project announcement about ...",0,ES2002,A
2,Designing a remote control.,0,ES2002,A
3,"That's about it, didn't get anything else.",0,ES2002,A
4,Did you get the same thing?,0,ES2002,A


In [5]:
!pip install -q -U \
  transformers \
  accelerate \
  bitsandbytes \
  peft \
  trl \
  datasets \
  scikit-learn \
  pandas \
  tqdm

In [6]:
import torch

print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "GPU memory:",
        round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2),
        "GB"
    )

GPU available: True
GPU: Tesla T4
GPU memory: 15.64 GB


# Prepare the Dataset

The dataset is divided into training, development, and test sets.  
Each row contains a meeting utterance and a binary label indicating whether the utterance is an action item.

To keep the experiments practical on a free Google Colab GPU, smaller representative subsets will be used for prompting and LoRA fine-tuning.

In [7]:
# Inspect the class distribution in each split.
# This helps us understand whether the dataset is balanced.

print("Training label distribution:")
print(train_df["label"].value_counts())

print("\nDevelopment label distribution:")
print(dev_df["label"].value_counts())

print("\nTest label distribution:")
print(test_df["label"].value_counts())

Training label distribution:
label
0    80825
1      255
Name: count, dtype: int64

Development label distribution:
label
0    17328
1       73
Name: count, dtype: int64

Test label distribution:
label
0    16854
1       53
Name: count, dtype: int64


In [46]:
# Use the full test set for evaluation

evaluation_df = test_df.copy()

print("Evaluation set shape:", evaluation_df.shape)

print("\nLabel distribution:")
print(evaluation_df["label"].value_counts())

Evaluation set shape: (16907, 4)

Label distribution:
label
0    16854
1       53
Name: count, dtype: int64


In [50]:
# Define the meaning of each class used by the language model.

LABEL_DESCRIPTIONS = {
    0: "Not an action item",
    1: "Action item"
}

evaluation_df["label_name"] = evaluation_df["label"].map(LABEL_DESCRIPTIONS)

evaluation_df[["text", "label", "label_name"]].head()

,text,label,label_name
0,Hmm hmm hmm.,0,Not an action item
1,Yeah.,0,Not an action item
2,"Okay. Yep, yep.",0,Not an action item
3,Okay.,0,Not an action item
4,Tu tu tu tu,0,Not an action item


Zero-Shot Prompting

The zero-shot prompting approach asks the language model to classify a meeting utterance without providing any labelled examples. The model must determine whether the utterance contains an actionable task based solely on the instruction.

In [51]:
# Function to create a zero-shot prompt.
# The language model must classify each meeting utterance
# without seeing any labelled examples.

def create_zero_shot_prompt(text):
    prompt = f"""
You are an expert assistant that identifies action items from meeting transcripts.

Definition:
An action item is a task, commitment, assignment, or follow-up that someone is expected to complete.

Instructions:
Read the meeting utterance carefully.

Return ONLY one label:

Action item
OR
Not an action item

Do not explain your answer.

Meeting utterance:
{text}

Classification:
"""
    return prompt

In [52]:
# Test the prompt
# Display an example prompt using the first evaluation sample.

example_prompt = create_zero_shot_prompt(
    evaluation_df.iloc[0]["text"]
)

print(example_prompt)


You are an expert assistant that identifies action items from meeting transcripts.

Definition:
An action item is a task, commitment, assignment, or follow-up that someone is expected to complete.

Instructions:
Read the meeting utterance carefully.

Return ONLY one label:

Action item
OR
Not an action item

Do not explain your answer.

Meeting utterance:
Hmm hmm hmm.

Classification:



Hugging Face Authentication

The Hugging Face Hub is used to access the gated Meta Llama model. Authentication is performed through a secure notebook prompt so that the access token is not stored directly in the notebook.

In [18]:
# Authenticate securely with Hugging Face.
# The token will be entered through a hidden input widget
# and will not appear in the notebook code.

from huggingface_hub import notebook_login

notebook_login()

Load the Meta Llama 3.2 Model

The instruction-tuned Meta Llama 3.2 model is loaded using 4-bit quantization. This reduces GPU memory requirements while maintaining good performance for inference and LoRA fine-tuning on the free Google Colab Tesla T4 GPU.

In [22]:
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)

import torch

In [26]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

In [23]:
# Name of the pretrained Meta Llama model.

MODEL_NAME = "meta-llama/Llama-3.2-3B-Instruct"

In [24]:
# Load the tokenizer.

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Ensure the tokenizer has a padding token.
tokenizer.pad_token = tokenizer.eos_token

config.json:   0%|          | 0.00/878 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/54.5k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

Load the Meta Llama 3.2 Model

The pretrained Meta Llama 3.2 3B Instruct model is loaded using 4-bit quantization. This reduces GPU memory consumption, making inference and LoRA fine-tuning feasible on the free Google Colab Tesla T4 GPU.

In [27]:
# Load the Meta Llama model using 4-bit quantization.

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
)

print("Model loaded successfully!")

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json:   0%|          | 0.00/20.9k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

Model loaded successfully!


Zero-Shot Inference

In this section, the Meta Llama 3.2 model is used to classify meeting utterances without any labelled examples (zero-shot prompting). The model predicts whether each utterance represents an action item based solely on the prompt instructions.

In [53]:
# Generate a prediction using the Meta Llama model.

def generate_prediction(prompt):

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=5,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )

    # Decode ONLY the newly generated tokens.
    generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]

    response = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    ).strip()

    return response

In [55]:
# Test
# Generate the first prediction.

example_text = evaluation_df.iloc[0]["text"]

prompt = create_zero_shot_prompt(example_text)

prediction = generate_prediction(prompt)

print(prediction)

Not an action item


In [57]:
# Extract only the predicted label from the model output.

def extract_prediction(response):

    if "Action item" in response and "Not an action item" not in response:
        return "Action item"

    elif "Not an action item" in response:
        return "Not an action item"

    return "Unknown"

In [58]:
# Test
example_text = evaluation_df.iloc[0]["text"]

prompt = create_zero_shot_prompt(example_text)

response = generate_prediction(prompt)

prediction = extract_prediction(response)

print("Prediction:", prediction)

Prediction: Not an action item


Zero-Shot Evaluation

The Meta Llama 3.2 model is evaluated using zero-shot prompting. Each utterance in the evaluation subset is classified without providing any labelled examples. The predicted labels are stored for quantitative evaluation.

In [38]:
# Import tqdm to display a progress bar during inference.

from tqdm.auto import tqdm

In [59]:
# Generate zero-shot predictions for every utterance
# in the evaluation dataset.

zero_shot_predictions = []

for text in tqdm(evaluation_df["text"], desc="Zero-shot inference"):

    prompt = create_zero_shot_prompt(text)

    response = generate_prediction(prompt)

    prediction = extract_prediction(response)

    zero_shot_predictions.append(prediction)

print("Zero-shot inference completed.")

Zero-shot inference:   0%|          | 0/16907 [00:00<?, ?it/s]

Zero-shot inference completed.


In [61]:
# Store the predictions in the evaluation dataframe.

evaluation_df["zero_shot_prediction"] = zero_shot_predictions

evaluation_df[
    ["text", "label_name", "zero_shot_prediction"]
].head(10)

,text,label_name,zero_shot_prediction
0,Hmm hmm hmm.,Not an action item,Not an action item
1,Yeah.,Not an action item,Not an action item
2,"Okay. Yep, yep.",Not an action item,Not an action item
3,Okay.,Not an action item,Not an action item
4,Tu tu tu tu,Not an action item,Not an action item
5,"Hi, good morning.",Not an action item,Not an action item
6,'Kay.,Not an action item,Not an action item
7,,Not an action item,Action item
8,Oops.,Not an action item,Not an action item
9,Mm.,Not an action item,Not an action item


Model Evaluation

The predictions generated by the Meta Llama 3.2 model are evaluated using standard binary classification metrics, including accuracy, precision, recall, and F1-score. These metrics provide a quantitative comparison with the RoBERTa baseline.

In [62]:
# Import evaluation metrics.

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

In [64]:
# Convert text predictions into binary labels.

prediction_map = {
    "Not an action item": 0,
    "Action item": 1
}

evaluation_df["predicted_label"] = (
    evaluation_df["zero_shot_prediction"]
    .map(prediction_map)
)

evaluation_df.head()

,text,label,meeting,speaker,label_name,zero_shot_prediction,predicted_label
0,Hmm hmm hmm.,0,ES2004,A,Not an action item,Not an action item,0.0
1,Yeah.,0,ES2004,A,Not an action item,Not an action item,0.0
2,"Okay. Yep, yep.",0,ES2004,A,Not an action item,Not an action item,0.0
3,Okay.,0,ES2004,A,Not an action item,Not an action item,0.0
4,Tu tu tu tu,0,ES2004,A,Not an action item,Not an action item,0.0


In [68]:
evaluation_df.loc[
    evaluation_df["zero_shot_prediction"] == "Unknown",
    ["text"]
].head(10)

,text
29,so basically these are the three unique featur...
791,and that's the number you put in
895,"Instruction manuals,"
930,"and, you don't need to send very much informat..."
1055,Just whack in the number.
1233,you get the idea.
1661,"what kind of size,"
3206,and be easy to locate
3861,If you hit just hit return and it should get r...
4204,and then sends the appropriate message to the ...


In [72]:
evaluation_df["predicted_label"] = (
    evaluation_df["predicted_label"]
    .fillna(0)
    .astype(int)
)
print(evaluation_df["predicted_label"].isna().sum())

0


In [74]:
evaluation_df["zero_shot_prediction"].value_counts().head(20)

,count
zero_shot_prediction,
Not an action item,11377
Action item,5495
Unknown,35


In [76]:
# Calculate evaluation metrics.

y_true = evaluation_df["label"]
y_pred = evaluation_df["predicted_label"]

accuracy = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred, zero_division=0)
recall = recall_score(y_true, y_pred, zero_division=0)
f1 = f1_score(y_true, y_pred, zero_division=0)

print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1-score : {f1:.4f}")

Accuracy : 0.6755
Precision: 0.0056
Recall   : 0.5849
F1-score : 0.0112


In [79]:
# Display the detailed classification report.

print(classification_report(y_true, y_pred))

              precision    recall  f1-score   support

           0       1.00      0.68      0.81     16854
           1       0.01      0.58      0.01        53

    accuracy                           0.68     16907
   macro avg       0.50      0.63      0.41     16907
weighted avg       0.99      0.68      0.80     16907



In [81]:
import json

results = {
    "model": "Meta LLaMA 3.2 3B Instruct (Zero-shot)",
    "accuracy": 0.6755,
    "precision": 0.0056,
    "recall": 0.5849,
    "f1": 0.0112
}

with open("llama_zero_shot_results.json", "w") as f:
    json.dump(results, f, indent=2)

print(results)

{'model': 'Meta LLaMA 3.2 3B Instruct (Zero-shot)', 'accuracy': 0.6755, 'precision': 0.0056, 'recall': 0.5849, 'f1': 0.0112}


Comparison with RoBERTa

| Model                             | Accuracy |  Precision |     Recall |   F1-score |
| --------------------------------- | -------: | ---------: | ---------: | ---------: |
| **RoBERTa (fine-tuned)**          |     0.68 | **0.3696** |     0.3208 | **0.3434** |
| **Meta LLaMA 3.2 3B (Zero-shot)** |   0.6755 |     0.0056 | **0.5849** |     0.0112 |


Comparison with RoBERTa: Both models were evaluated on the same test set of 16,907 meeting utterances. The fine-tuned RoBERTa model outperformed the zero-shot LLaMA model, achieving a much higher precision (0.3696 vs. 0.0056) and F1-score (0.3434 vs. 0.0112). Although LLaMA achieved higher recall (0.5849 vs. 0.3208), it generated many false positives by incorrectly classifying non-action items as action items. These results indicate that fine-tuning on the task is more effective than zero-shot prompting for action item detection in meeting transcripts.

Error Analysis

In [82]:
# Find the incorrect predictions

incorrect = evaluation_df[
    evaluation_df["label"] != evaluation_df["predicted_label"]
]

print("Number of incorrect predictions:", len(incorrect))

incorrect[
    ["text", "label_name", "zero_shot_prediction"]
].head(10)

Number of incorrect predictions: 5486


,text,label_name,zero_shot_prediction
7,,Not an action item,Action item
12,"Yeah, me.",Not an action item,Action item
14,Where did this come from?,Not an action item,Action item
17,"Uh, maybe you can guess what I'm trying to make?",Not an action item,Action item
21,"Okay, I see it as one thing it's very supportive.",Not an action item,Action item
24,"it can be your best friend,",Not an action item,Action item
25,"it doesn't discriminate between you, based on ...",Not an action item,Action item
35,,Not an action item,Action item
36,"Eagle, okay.",Not an action item,Action item
37,One point four or something like that.,Not an action item,Action item


Error Analysis

The zero-shot LLaMA model correctly classified 11,421 out of 16,907 utterances and incorrectly classified 5,486. Most errors were false positives, where normal conversational utterances were predicted as action items. This indicates that the model tends to over-predict action items when using zero-shot prompting, resulting in high recall but very low precision.

In [84]:
# Find the correct predictions

correct = evaluation_df[
    evaluation_df["label"] == evaluation_df["predicted_label"]
]

print("Number of correct predictions:", len(correct))

correct[
    ["text", "label_name", "zero_shot_prediction"]
].head(10)

Number of correct predictions: 11421


,text,label_name,zero_shot_prediction
0,Hmm hmm hmm.,Not an action item,Not an action item
1,Yeah.,Not an action item,Not an action item
2,"Okay. Yep, yep.",Not an action item,Not an action item
3,Okay.,Not an action item,Not an action item
4,Tu tu tu tu,Not an action item,Not an action item
5,"Hi, good morning.",Not an action item,Not an action item
6,'Kay.,Not an action item,Not an action item
8,Oops.,Not an action item,Not an action item
9,Mm.,Not an action item,Not an action item
10,Oh sorry.,Not an action item,Not an action item


Methodology

We evaluated the Meta LLaMA 3.2 3B Instruct model using a zero-shot prompting approach for action item detection. Instead of fine-tuning the model, we provided an instruction prompt defining an action item and asked the model to classify each meeting utterance as either "Action item" or "Not an action item."

The model was evaluated on the AMI Meeting Corpus test split containing 16,907 utterances. Predictions were converted into binary labels and compared with the ground-truth annotations. Model performance was measured using accuracy, precision, recall, and F1-score, allowing a direct comparison with the fine-tuned RoBERTa baseline.

In [85]:
from google.colab import files
files.download("llama_zero_shot_results.json")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

git add Agnes_LLaMA_Evaluation.ipynb llama_zero_shot_results.json

git commit -m "Add LLaMA zero-shot evaluation"

git push